# Magnus Carlsen Performance Analysis — ML Prediction

Building on the cleaned dataset and patterns surfaced during EDA, particularly the 
observed within session accuracy decay, this notebook tests whether Magnus's accuracy 
in his *next* game can be predicted from pre-game context: session fatigue signals, 
recent form, rating gap, and opponent strength.

A RandomForestRegressor is trained to predict `next_game_accuracy`, using a 
sklearn Pipeline for leakage-safe preprocessing. The notebook reports model 
performance honestly, including where the model's predictive power is limited; 
since much of in-game accuracy is driven by move-by-move content the pregame 
features can't capture.

## Dataset Selection: Standard Chess vs. Unfiltered Games

We import `magnus_cleaned_chess_games` (standard chess only) directly instead of `magnus_cleaned_games` for two practical reasons:

1. **Avoids Structural `NaN`s in `opening_simple`:** Chess variants like Chess960 start with randomized piece setups and have no standard opening classifications, resulting in missing `opening_simple` values. Importing the pre-filtered standard chess dataset removes these variant games naturally up front, avoiding unnecessary `NaN` imputation or manual row dropping during data cleaning.
2. **Model Consistency:** Features like opening types, player accuracy, and historical performance follow traditional opening theory. Excluding non-standard variants ensures that the predictive model learns patterns strictly from standard chess gameplay.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.inspection import permutation_importance

In [2]:
games = pd.read_csv("magnus_cleaned_chess_games.csv")

In [3]:
games.shape

(8951, 34)

---
## The target

Predicting magnus_accuracy shifted –1 within session groups; in plain words: given a game Magnus just finished, predict his accuracy in the next game of that same session, with the last game of each session dropped (since it has no "next game" to predict).

>This came out of the within-session accuracy decay pattern you found in EDA; the idea that his accuracy tends to drift as a session goes on (fatigue), and the model is testing whether that's predictable.

In [4]:
games = games.sort_values('end_time').reset_index(drop=True)   # sort BEFORE shifting
games['target'] = games.groupby('session_id')['magnus_accuracy'].shift(-1)
games = games.dropna(subset=['target'])

In [5]:
games.loc[8940:,['magnus_accuracy','target']]

,magnus_accuracy,target
8940,91.37,93.10
8941,93.10,90.53
8942,90.53,93.15
8943,93.15,86.40
8944,86.40,90.42
8945,90.42,94.23
8947,89.88,81.37
8948,81.37,96.35
8949,96.35,95.80


---
## Features

time_class, magnus_result_simple, opening_simple, year,
game_number_in_session, magnus_accuracy (current game),
opponent_accuracy, magnus_rating, opponent_rating, rating_diff,
acc_diff, magnus_color, base_time_seconds, increment_seconds, time_since_last_game_min, rolling_avg_accuracy_last5 (shift(1).rolling(5).mean())

In [6]:
games.isnull().sum()

url                         0
time_class                  0
time_control                0
rated                       0
rules                       0
end_time                    0
eco_url                     0
opening                     0
white_username              0
white_rating                0
white_result                0
black_username              0
black_rating                0
black_result                0
magnus_color                0
magnus_rating               0
opponent_rating             0
magnus_result               0
white_accuracy              0
black_accuracy              0
magnus_accuracy             0
date                        0
year                        0
magnus_result_simple        0
prev_result                 0
opening_simple              0
opponent_accuracy           0
game_type                   0
time_since_last_game_min    1
is_new_session              0
session_id                  0
game_number_in_session      0
rating_diff                 0
is_upset  

In [7]:
games.columns

Index(['url', 'time_class', 'time_control', 'rated', 'rules', 'end_time',
       'eco_url', 'opening', 'white_username', 'white_rating', 'white_result',
       'black_username', 'black_rating', 'black_result', 'magnus_color',
       'magnus_rating', 'opponent_rating', 'magnus_result', 'white_accuracy',
       'black_accuracy', 'magnus_accuracy', 'date', 'year',
       'magnus_result_simple', 'prev_result', 'opening_simple',
       'opponent_accuracy', 'game_type', 'time_since_last_game_min',
       'is_new_session', 'session_id', 'game_number_in_session', 'rating_diff',
       'is_upset', 'target'],
      dtype='object')

---
## Handling Missing values in fetures

In [8]:
games['time_since_last_game_min'] = games['time_since_last_game_min'].fillna(0)

In [9]:
games.shape

(8063, 35)

---
## Feature Engineering

### 1. 'acc_diff'

In [10]:
games['acc_diff'] = games['magnus_accuracy']-games['opponent_accuracy']

In [11]:
games.columns[-1]

'acc_diff'

### 2. 'time_control' to 'base_time_seconds' and 'increment_seconds'

In [12]:
def parse_time_control(tc):
    if '+' in str(tc):
        base, inc = tc.split('+')
        return int(base), int(inc)
    else:
        return int(tc), 0

games[['base_time_seconds', 'increment_seconds']] = games['time_control'].apply(
    lambda tc: pd.Series(parse_time_control(tc))
)

In [13]:
games[['base_time_seconds', 'increment_seconds','time_control']]

,base_time_seconds,increment_seconds,time_control
0,1500,25,1500+25
1,1500,25,1500+25
2,1500,25,1500+25
4,300,2,300+2
5,300,2,300+2
...,...,...,...
8944,180,0,180
8945,180,0,180
8947,180,0,180
8948,180,0,180


In [14]:
#verification

print(
    games[games['base_time_seconds'].isnull()].index, "\n",
    games[games['increment_seconds'].isnull()].index 
     )

Index([], dtype='int64') 
 Index([], dtype='int64')


### 3. 'rolling_avg_accuracy_last5'

In [15]:
#ensuring the dataset is sorted chronologically
games = games.sort_values('end_time').reset_index(drop=True)

#computing in session rolling average (last5 games,shifted by 1 to prevent leakage)
in_session_rolling = games.groupby('session_id')['magnus_accuracy'].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
)

#get immediate previous game accuracy across the entire dataset
previous_game_accuracy = games['magnus_accuracy'].shift(1)

#impute Game 1 NaNs with previous session's last game accuracy
games['rolling_avg_accuracy_last5'] = in_session_rolling.fillna(previous_game_accuracy)

#imputing index 0 (very first game in dataset history) is still left which we can do with
'''
games['rolling_avg_accuracy_last5'] = games['rolling_avg_accuracy_last5'].fillna(
    games['magnus_accuracy'].mean()
)
''' #But is a form of data leak as the mean would include the games in test split

"\ngames['rolling_avg_accuracy_last5'] = games['rolling_avg_accuracy_last5'].fillna(\n    games['magnus_accuracy'].mean()\n)\n"

In [16]:
games.columns[-1]

'rolling_avg_accuracy_last5'

In [17]:
#Verification

print(f"Missing values in rolling_avg_accuracy_last5: {games['rolling_avg_accuracy_last5'].isna().sum()}")

Missing values in rolling_avg_accuracy_last5: 1


### This missing value is at index 0, basically in train split, so we will impute it later with the mean of train split

---
## Data Split

Making sure data is sorted since our analysis holds together upon sequential data

In [18]:
games['end_time'].is_monotonic_increasing

True

In [19]:
features = [
            'time_class', 'magnus_result_simple', 'opening_simple', 'game_number_in_session',
            'magnus_accuracy', 'opponent_accuracy', 'time_since_last_game_min', 'year',
            'magnus_rating', 'opponent_rating', 'rating_diff', 'acc_diff','magnus_color',
            'base_time_seconds','increment_seconds','rolling_avg_accuracy_last5'
           ]
X = games[features]
y = games['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

In [20]:
X_train[X_train['rolling_avg_accuracy_last5'].isnull()].index

Index([0], dtype='int64')

### To prevent data leakage we will take the mean of just train split to fix NaN in train split

In [21]:
# 1. Compute training set mean strictly from X_train
train_mean_accuracy = X_train['magnus_accuracy'].mean()

# 2. Impute Index 0 of X_train with the train mean
X_train['rolling_avg_accuracy_last5'] = X_train['rolling_avg_accuracy_last5'].fillna(train_mean_accuracy)

# Note: X_test won't have NaNs because its first game seamlessly references 
# the previous game from X_train (prior history carrying over).

In [22]:
X_train[X_train['rolling_avg_accuracy_last5'].isnull()].index

Index([], dtype='int64')

---
## Preprocessor 

In [23]:
#feature groups
ordinal_cols = ['time_class'] 
ohe_cols = ['opening_simple', 'magnus_result_simple', 'magnus_color']  
#Automatically identify numeric columns basically all remaining columns except categorical ones
numeric_cols = [c for c in X_train.columns if c not in ordinal_cols + ohe_cols]

#Define explicit order for ordinal feature
#fastest to slowest
time_class_order = [['bullet', 'blitz', 'rapid']]

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'ord', 
            OrdinalEncoder(
                categories=time_class_order, 
                handle_unknown='use_encoded_value', 
                unknown_value=-1
            ), 
            ordinal_cols
        ),
        (
            'ohe', 
            OneHotEncoder(
                drop=None,  # Recommended for Random Forest
                sparse_output=False, 
                handle_unknown='ignore'
            ), 
            ohe_cols
        )
    ],
    remainder='passthrough'  # CRITICAL: Keeps all numeric features untouched
)

---
## Pipeline

### 1. Hyperparameter Search (Time-Series Cross-Validation)

-  TimeSeriesSplit ensures each fold only trains on PAST data, validates on FUTURE data
-  plain cv=5 would leak future games into training, since our data is chronological.
-  This loop only ever touches X_train / y_train — X_test stays untouched.

In [25]:

tscv = TimeSeriesSplit(n_splits=5)

n_estimators_list = [100, 200, 300]
max_depth_list = [1, 2, 3, None]

best_cv_rmse = float('inf')
best_params = {}

for n in n_estimators_list:
    for depth in max_depth_list:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=42, n_jobs=-1)
        test_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', rf)])

        scores = cross_val_score(
            test_pipeline, X_train, y_train,
            cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
        )
        cv_rmse = -scores.mean()

        print(f"n_estimators={n:<4} | max_depth={str(depth):<5} | CV RMSE: {cv_rmse:.4f}")

        if cv_rmse < best_cv_rmse:
            best_cv_rmse = cv_rmse
            best_params = {'n_estimators': n, 'max_depth': depth}

print(f"\nBest Combination Found: {best_params} (CV RMSE: {best_cv_rmse:.4f})")

n_estimators=100  | max_depth=1     | CV RMSE: 6.2355
n_estimators=100  | max_depth=2     | CV RMSE: 6.1952
n_estimators=100  | max_depth=3     | CV RMSE: 6.2175
n_estimators=100  | max_depth=None  | CV RMSE: 6.4535
n_estimators=200  | max_depth=1     | CV RMSE: 6.2358
n_estimators=200  | max_depth=2     | CV RMSE: 6.1942
n_estimators=200  | max_depth=3     | CV RMSE: 6.2102
n_estimators=200  | max_depth=None  | CV RMSE: 6.4320
n_estimators=300  | max_depth=1     | CV RMSE: 6.2387
n_estimators=300  | max_depth=2     | CV RMSE: 6.1957
n_estimators=300  | max_depth=3     | CV RMSE: 6.2106
n_estimators=300  | max_depth=None  | CV RMSE: 6.4261

Best Combination Found: {'n_estimators': 200, 'max_depth': 2} (CV RMSE: 6.1942)


### 2. Final Model — Fit on Full Training Set
- Best hyperparameters were chosen using ONLY X_train/y_train (via CV above).
- Now train one final pipeline on the complete training set.

In [26]:
final_rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
final_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', final_rf)])
final_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ord', ...), ('ohe', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### 3. Evaluation — Train vs. Test (held-out, touched once)
- Comparing train vs. test R² reveals whether the model is overfitting
- (high train R², much lower test R²) or generalizing (both modest and close).

In [27]:
from sklearn.metrics import root_mean_squared_error, r2_score

train_preds = final_pipeline.predict(X_train)
test_preds  = final_pipeline.predict(X_test)

train_rmse = root_mean_squared_error(y_train, train_preds)
test_rmse  = root_mean_squared_error(y_test, test_preds)
train_r2   = r2_score(y_train, train_preds)
test_r2    = r2_score(y_test, test_preds)

print(f"Train RMSE: {train_rmse:.4f}  |  Train R²: {train_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.4f}  |  Test  R²: {test_r2:.4f}")

Train RMSE: 6.1733  |  Train R²: 0.0649
Test  RMSE: 6.2143  |  Test  R²: 0.0554


### 4. Feature Importances
- Get feature names after preprocessing (one-hot expands opening_simple into many columns)


In [28]:
feature_names = final_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = final_pipeline.named_steps['regressor'].feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

importance_df.head(15)

,feature,importance
79,remainder__time_since_last_game_min,0.521213
82,remainder__opponent_rating,0.135668
85,remainder__base_time_seconds,0.125397
0,ord__time_class,0.075410
86,remainder__increment_seconds,0.054171
24,ohe__opening_simple_Grob Opening,0.036110
83,remainder__rating_diff,0.018518
81,remainder__magnus_rating,0.009375
87,remainder__rolling_avg_accuracy_last5,0.009302
77,remainder__magnus_accuracy,0.008586


In [29]:
perm = permutation_importance(
    final_pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': X_test.columns,  # permutation_importance uses pre-encoding column names
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

perm_df.head(15)

,feature,importance_mean,importance_std
6,time_since_last_game_min,4.957737e-02,6.767344e-03
13,base_time_seconds,1.038765e-02,1.694147e-03
0,time_class,3.786228e-03,1.167391e-03
9,opponent_rating,1.849608e-03,1.577980e-03
10,rating_diff,5.587791e-04,3.149192e-04
15,rolling_avg_accuracy_last5,5.327181e-04,2.239770e-04
4,magnus_accuracy,4.611701e-05,1.465779e-04
5,opponent_accuracy,3.027773e-05,3.027773e-05
2,opening_simple,7.859503e-06,2.484906e-04
12,magnus_color,2.331468e-16,2.352520e-16


---
---
# CONCLUSION










## What the model found

The final model explains a modest but real amount of variance in next-game 
accuracy (test R² ≈ 0.055) — roughly 5.5% of the variation, meaning its 
predictions are only slightly better than guessing Magnus's average accuracy 
every time. The hyperparameter search reinforces this: the best model settled 
on `max_depth=2`, an almost trivially shallow tree, and allowing deeper trees 
(`max_depth=None`) made cross-validated error *worse* — the algorithm wasn't 
finding deeper patterns, it was fitting noise.

Comparing impurity-based and permutation importance was informative in its own 
right. Impurity importance initially suggested `opponent_rating` was a major 
driver, but permutation importance — which measures the actual drop in test 
performance when a feature is shuffled — showed this was misleading; shuffling 
`opponent_rating` barely moves prediction error. The feature that holds up 
under both methods is `time_since_last_game_min`, with `base_time_seconds` a 
distant but real second. Every other feature — including the ones the original 
fatigue hypothesis was built around (`game_number_in_session`, 
`rolling_avg_accuracy_last5`) — contributed negligibly once measured correctly.



## Why this result makes sense

This result isn't a failure of the model or the feature engineering — it's 
consistent with what the dataset can and can't represent. A chess game's 
accuracy is decided move by move, inside the game: the specific position 
reached, whether a tactic was missed or found, how sharp or simplified the 
position became, how much the opponent's choices forced difficult decisions. 
None of that lives in this dataset. What this dataset holds is *post-game and 
pre-game context* — ratings, time formats, outcomes, timestamps — a layer 
that sits entirely outside the game itself.

Put directly: predicting an in-game outcome (accuracy) from pre-game and 
post-game metadata means asking context about a game to substitute for 
content within it. Context can set the conditions a game is played under, but 
it can't see the moves. A session-fatigue signal might nudge the odds of a 
mistake, but it can't tell you whether that mistake actually got made, missed, 
or punished in this specific game — that depends on what's on the board.

Seen this way, the weak R² isn't an open question so much as an expected 
outcome, now confirmed rather than assumed: this feature set was always 
structurally one level removed from what actually determines the target.



## Where real signal would have to come from

Closing this gap would likely require in-game features this dataset doesn't 
contain — engine evaluation swings move by move, time spent per move, how 
sharp or forcing the position was, or move-level complexity. Those require 
PGN-level analysis, not just game metadata, and are a natural direction for a 
follow-up notebook rather than further tuning of this one. Further grid 
search or feature engineering within the current feature set is unlikely to 
move this result meaningfully — the ceiling here is the data available, not 
the model.

----
<div align="center">
   X
</div>

---